# Trajectory Prediction Demo

Import dependencies, load processed data, split, train LSTM, evaluate metrics, and visualize compared with Kalman.

In [6]:
import os
import sys
import glob
import random
import torch
import numpy as np
import matplotlib.pyplot as plt

# 将仓库根加入 sys.path（notebook 通常从仓库根启动；从 notebooks/ 启动时自动上移一层）
ROOT = os.getcwd()
if os.path.basename(ROOT) == "notebooks":
    ROOT = os.path.dirname(ROOT)
if ROOT not in sys.path:
    sys.path.insert(0, ROOT)

from models.learning_based.lstm_predictor import LSTMTrajectoryModel
from models.rule_based.kalman_predictor import KalmanTrajectoryPredictor
from models.loss_common import ade_loss, fde_loss, WeightedSmoothL1Loss
from models.metrics_common import compute_ade, compute_fde
from scripts.preprocess.argo1_dataset import ArgoverseV1Dataset


In [ ]:
# 根据实际数据修改：原始数据根目录（Argoverse 1 官方布局 data/raw/{train,val}/data）
root = ROOT
raw_data_dir = os.path.join(root, "data", "raw")     # 原始 CSV 目录
processed_dir = os.path.join(root, "data", "processed")  # 预处理输出目录
if not os.path.isdir(processed_dir) or len(glob.glob(os.path.join(processed_dir, "*.pt"))) == 0:
    raw_csvs = glob.glob(os.path.join(raw_data_dir, "*.csv"))
    if not raw_csvs:
        raise FileNotFoundError(
            f"No .csv files directly under {raw_data_dir}. Argoverse 1 official "
            f"layout is data/raw/{{train,val}}/data/*.csv — point raw_data_dir at "
            f"the directory that directly contains the scene CSVs (e.g. data/raw/val/data)."
        )
    ds = ArgoverseV1Dataset(root, raw_dir=raw_data_dir, processed_dir=processed_dir)
    ds.process()
files = sorted(glob.glob(os.path.join(processed_dir, "*.pt")))
random.seed(42)
random.shuffle(files)
files = files[:2000]  # 演示用 2000 个，可改为全部数据

In [ ]:
import glob, os

def load_sample(path):
    d = torch.load(path, weights_only=False)
    idx = int(d["agent_index"]) if "agent_index" in d else 0
    hist = d["x"][idx]
    fut = d["y"][idx]
    return hist, fut

# Split: prefer official split subdirectories (processed_dir/train, /val) when
# present; otherwise fall back to a random 80/20 file split and warn — a random
# split of one pool is NOT the official split, so if the pool mixes official
# train and val scenes, validation metrics would be optimistic (leak).
train_dir = os.path.join(processed_dir, "train")
val_dir = os.path.join(processed_dir, "val")
train_files = sorted(glob.glob(os.path.join(train_dir, "*.pt")))
val_files = sorted(glob.glob(os.path.join(val_dir, "*.pt")))
if train_files and val_files:
    print(f"Using official split dirs: train={len(train_files)}, val={len(val_files)}")
else:
    split = int(len(files) * 0.8)
    train_files = files[:split]
    val_files = files[split:]
    print("WARN: no processed_dir/train+val subdirs found — using a random 80/20 file split.")
    print("      If this pool mixes official train AND val scenes, validation is leaked.")
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

In [9]:
m = LSTMTrajectoryModel(input_size=2, hidden_size=128, num_layers=2, dropout=0.5).to(device)
opt = torch.optim.Adam(m.parameters(), lr=1e-3)
crit = WeightedSmoothL1Loss(beta=1.0, reduction="mean")

In [10]:
epochs = 10  # 10 rounds for demo ,you can change it for better performance
train_subset = train_files[:1000]
for ep in range(epochs):
    m.train()
    total = 0.0
    for p in train_subset:
        hist, fut = load_sample(p)
        hist = hist.to(device)
        fut = fut.to(device)
        pred = m(hist.unsqueeze(0), steps=fut.shape[0]).squeeze(0)
        loss = crit(pred.unsqueeze(0), fut.unsqueeze(0)) + 0.5 * fde_loss(pred.unsqueeze(0), fut.unsqueeze(0))
        opt.zero_grad()
        loss.backward()
        opt.step()
        total += loss.item()
    avg = total / max(1, len(train_subset))
    print("epoch", ep, "train_loss", round(avg, 4))

In [11]:
m.eval()
ades = []
fdes = []
for p in val_files[:200]:
    hist, fut = load_sample(p)
    with torch.no_grad():
        pred = m(hist.to(device).unsqueeze(0), steps=fut.shape[0]).squeeze(0).cpu()
    ade = compute_ade(pred.unsqueeze(0), fut.unsqueeze(0))
    fde = compute_fde(pred.unsqueeze(0), fut.unsqueeze(0))
    ades.append(ade)
    fdes.append(fde)
print("LSTM ADE", float(np.mean(ades)))
print("LSTM FDE", float(np.mean(fdes)))

In [12]:
def visualize_sample(index):
    if index >= len(val_files):
        print(f'Index {index} out of range (max {len(val_files)-1})')
        return
    
    path = val_files[index]
    # Load full dictionary to get metadata
    d = torch.load(path, weights_only=False)
    seq_id = d['seq_id']
    agent_idx = int(d['agent_index'])
    
    print(f'Visualizing sample index: {index}')
    print(f'File path: {path}')
    print(f'Sequence ID: {seq_id}')
    
    hist = d['x'][agent_idx]
    fut = d['y'][agent_idx]
    
    # Kalman Prediction
    kal = KalmanTrajectoryPredictor(dt=1.0).fit(hist.numpy())
    kal_pred = torch.tensor(kal.forecast(fut.shape[0]), dtype=torch.float32)
    
    # LSTM Prediction
    with torch.no_grad():
        lstm_pred = m(hist.to(device).unsqueeze(0), steps=fut.shape[0]).squeeze(0).cpu()
        
    # Metrics
    kal_ade = compute_ade(kal_pred.unsqueeze(0), fut.unsqueeze(0))
    kal_fde = compute_fde(kal_pred.unsqueeze(0), fut.unsqueeze(0))
    lstm_ade = compute_ade(lstm_pred.unsqueeze(0), fut.unsqueeze(0))
    lstm_fde = compute_fde(lstm_pred.unsqueeze(0), fut.unsqueeze(0))
    
    print(f'Kalman: ADE={kal_ade:.4f}, FDE={kal_fde:.4f}')
    print(f'LSTM:   ADE={lstm_ade:.4f}, FDE={lstm_fde:.4f}')
    
    import matplotlib
    fig, ax = plt.subplots(figsize=(10,10))
    
    # Plot trajectories
    ax.plot(hist[1:,0].numpy(), hist[1:,1].numpy(), 'k--', marker='.', label='History')
    ax.plot(fut[1:,0].numpy(), fut[1:,1].numpy(), 'g-', marker='.', label='Ground Truth')
    ax.plot(lstm_pred[:,0].numpy(), lstm_pred[:,1].numpy(), 'b-', marker='x', label=f'LSTM (ADE={lstm_ade:.2f})')
    ax.plot(kal_pred[:,0].numpy(), kal_pred[:,1].numpy(), 'r-', marker='x', label=f'Kalman (ADE={kal_ade:.2f})')
    
    # Draw Agent Rectangle with Heading
    if len(hist) >= 2:
        current_pos = hist[-1].numpy()
        prev_pos = hist[-2].numpy()
        heading = np.arctan2(current_pos[1] - prev_pos[1], current_pos[0] - prev_pos[0])
        
        # Vehicle dimensions (approximate)
        length = 4.0
        width = 2.0
        
        # Create rectangle centered at current_pos
        # We create it at origin then transform
        rect = plt.Rectangle((-length/2, -width/2), length, width, color='orangered', alpha=0.8, label='Agent')
        
        # Transform: Rotate then Translate
        t = matplotlib.transforms.Affine2D().rotate(heading).translate(current_pos[0], current_pos[1]) + ax.transData
        rect.set_transform(t)
        ax.add_patch(rect)
        
        # Add arrow to show direction clearly
        arrow_len = 3.0
        ax.arrow(current_pos[0], current_pos[1], 
                 arrow_len * np.cos(heading), arrow_len * np.sin(heading), 
                 head_width=0.5, head_length=0.5, fc='orangered', ec='orangered')
    
    ax.axis('equal')
    ax.legend()
    ax.set_title(f'Trajectory Prediction (Seq: {seq_id}, Val Index: {index})')
    ax.grid(True, alpha=0.3)
    plt.show()

In [13]:
# 修改 index 以可视化不同样本
visualize_sample(200)  ## 默认第 200 个样本
